# Task 121: neutompy — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Neutron tomography reconstruction using FBP (duplicate)

**Paper**: NeuTomPy toolbox, a Python package for tomographic data processing and reconstruction
**Repository**: https://github.com/dmici/NeuTomPy-toolbox

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 31.54 dB
- **SSIM**: 0.6453
- **Evaluation**: Neutron tomography 2D/3D reconstruction via FBP (PSNR + SSIM + RMSE)

### Mathematical Formulation

The Radon transform maps the 2D attenuation $f(x,y)$ to sinogram $p(\theta, s)$:

$$p(\theta, s) = \int_{-\infty}^{\infty} \int_{-\infty}^{\infty} f(x,y)\,\delta(x\cos\theta + y\sin\theta - s)\,dx\,dy$$

**Filtered Back-Projection (FBP)**:
$$f(x,y) = \int_0^{\pi} \left[ p(\theta, \cdot) \ast g \right](x\cos\theta + y\sin\theta)\,d\theta$$

where $g$ is the ramp filter: $\hat{g}(\omega) = |\omega|$.

**Iterative reconstruction** (SIRT/CGLS):
$$x^{(k+1)} = x^{(k)} + \alpha \, A^T(b - Ax^{(k)})$$

**TV-regularized**:
$$\hat{x} = \arg\min_x \frac{1}{2}\|Ax - b\|_2^2 + \lambda \|\nabla x\|_1$$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
neutompy — Neutron Tomography Reconstruction
=============================================
From neutron transmission radiographs at multiple angles, reconstruct the 2D
attenuation coefficient distribution using Filtered Backprojection (FBP).

Physics:
  - Forward: Beer-Lambert law: I(θ,s) = I_0 × exp(-∫ μ(x,y) dl)
    → sinogram: p(θ,s) = -ln(I/I_0) = Radon transform of μ(x,y)
  - Inverse: Filtered Backprojection with Ram-Lak filter
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`main`

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 6. MAIN
# ═══════════════════════════════════════════════════════════════════
def main():
    print("=" * 60)
    print("neutompy — Neutron Tomography Reconstruction")
    print("=" * 60)

    # 1. Create ground truth
    print("[1/4] Creating phantom (attenuation map) ...")
    gt = create_phantom(N)

    # 2. Forward model
    print(f"[2/4] Simulating neutron transmission ({N_ANGLES} angles) ...")
    sinogram, sinogram_ideal, angles = forward_neutron_tomo(gt, N_ANGLES, NOISE_LEVEL)

    # 3. FBP reconstruction
    print("[3/4] Running Filtered Backprojection ...")
    recon = fbp_reconstruct(sinogram, angles)

    # 4. Metrics
    metrics = compute_metrics(gt, recon)
    print(f"  PSNR = {metrics['PSNR']:.2f} dB")
    print(f"  SSIM = {metrics['SSIM']:.4f}")
    print(f"  RMSE = {metrics['RMSE']:.4f}")
    print(f"  CC   = {metrics['CC']:.4f}")

    # 5. Save
    print("[4/4] Saving results ...")
    for d in [RESULTS_DIR, ASSETS_DIR]:
        os.makedirs(d, exist_ok=True)
        np.save(os.path.join(d, "gt_output.npy"), gt)
        np.save(os.path.join(d, "recon_output.npy"), recon)
        np.save(os.path.join(d, "ground_truth.npy"), gt)
        np.save(os.path.join(d, "reconstruction.npy"), recon)
        with open(os.path.join(d, "metrics.json"), "w") as f:
            json.dump(metrics, f, indent=2)

    visualize(gt, sinogram, recon, metrics)

    print("Done ✓")
    return metrics

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `create_phantom`, `forward_neutron_tomo`

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 1. GROUND TRUTH: 2D attenuation coefficient phantom
# ═══════════════════════════════════════════════════════════════════
def create_phantom(n):
    """
    Create a 2D cross-section phantom mimicking metal cylinders embedded
    in a matrix material (typical neutron tomography sample).

    Attenuation coefficients per pixel (dimensionless).
    Scaled so that max line integral through the sample ≈ 3-5,
    giving meaningful Beer-Lambert transmission.
    
    Physical analogy (thermal neutrons, ~10cm sample on 256 pixels):
      pixel_size ≈ 0.04 cm → μ_pixel = μ_physical × pixel_size
      - Air/void:    0
      - Aluminum:    μ ≈ 0.1 cm⁻¹  → 0.004/pixel
      - Steel:       μ ≈ 1.0 cm⁻¹  → 0.04/pixel
      - Water:       μ ≈ 3.5 cm⁻¹  → 0.14/pixel
      - Lead:        μ ≈ 0.4 cm⁻¹  → 0.016/pixel
    """
    phantom = np.zeros((n, n), dtype=np.float64)
    yy, xx = np.ogrid[:n, :n]
    c = n // 2

    # Scale factor: physical_mu * pixel_size
    s = 0.04  # pixel size in cm

    # Outer aluminum casing (cylinder)
    r_outer = np.sqrt((yy - c)**2 + (xx - c)**2)
    phantom[r_outer < 0.42 * n] = 0.1 * s   # aluminum matrix

    # Steel cylinder 1 (off-center)
    r1 = np.sqrt((yy - c + 30)**2 + (xx - c - 40)**2)
    phantom[r1 < 25] = 1.0 * s

    # Steel cylinder 2
    r2 = np.sqrt((yy - c - 35)**2 + (xx - c + 25)**2)
    phantom[r2 < 20] = 1.0 * s

    # Water-filled cavity (high attenuation for neutrons)
    r3 = np.sqrt((yy - c + 10)**2 + (xx - c + 10)**2)
    phantom[r3 < 15] = 3.5 * s

    # Lead inclusion (moderate attenuation)
    r4 = np.sqrt((yy - c - 20)**2 + (xx - c - 30)**2)
    phantom[r4 < 12] = 0.4 * s

    # Small void (crack/pore)
    r5 = np.sqrt((yy - c + 40)**2 + (xx - c + 50)**2)
    phantom[r5 < 8] = 0.0

    # Another small steel rod
    r6 = np.sqrt((yy - c - 50)**2 + (xx - c - 10)**2)
    phantom[r6 < 10] = 0.8 * s

    # Annular structure
    r7 = np.sqrt((yy - c + 45)**2 + (xx - c - 15)**2)
    phantom[(r7 > 12) & (r7 < 18)] = 1.2 * s

    return phantom

# ═══════════════════════════════════════════════════════════════════
# 2. FORWARD MODEL: Beer-Lambert + Radon
# ═══════════════════════════════════════════════════════════════════
def forward_neutron_tomo(phantom, n_angles, noise_level):
    """
    Simulate neutron transmission radiography.

    Physical process:
    1. Neutron beam traverses the sample: line integrals of μ(x,y)
    2. Beer-Lambert law: I = I_0 exp(-∫μ dl) → transmission image
    3. Poisson counting noise corrupts transmission
    4. Reconstruction works on sinogram = -ln(I/I_0) ≈ Radon transform + noise

    For the simulation, we:
    1. Compute ideal sinogram via Radon transform
    2. Add realistic noise (Gaussian approximation of Poisson in log domain)
    """
    angles = np.linspace(0, 180, n_angles, endpoint=False)

    # Radon transform → sinogram (ideal line integrals of μ)
    sinogram_ideal = radon(phantom, theta=angles, circle=False)

    # Beer-Lambert noise model:
    # I = I0 * exp(-sinogram) → Poisson noise → -ln(I_noisy / I0)
    transmitted = I0 * np.exp(-sinogram_ideal)
    transmitted_noisy = np.random.poisson(
        np.maximum(transmitted, 1).astype(np.float64)
    ).astype(np.float64)
    # Add Gaussian readout noise (small)
    transmitted_noisy += np.random.normal(0, 2.0, transmitted_noisy.shape)
    transmitted_noisy = np.maximum(transmitted_noisy, 1.0)  # avoid log(0)
    sinogram_noisy = -np.log(transmitted_noisy / I0)

    return sinogram_noisy, sinogram_ideal, angles

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `fbp_reconstruct`

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 3. INVERSE: Filtered Backprojection
# ═══════════════════════════════════════════════════════════════════
def fbp_reconstruct(sinogram, angles):
    """
    Filtered Backprojection (FBP) with Ram-Lak filter.
    
    Note: We pass the sinogram directly (already as line-integral data).
    skimage.transform.iradon expects sinogram = Radon transform output.
    """
    recon = iradon(sinogram, theta=angles, filter_name="ramp", circle=False)
    recon = np.maximum(recon, 0)  # physical: attenuation >= 0
    return recon

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `visualize`

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# 4. METRICS
# ═══════════════════════════════════════════════════════════════════
def compute_metrics(gt, recon):
    """Compute PSNR, SSIM, RMSE."""
    # Crop to same size (iradon may produce slightly different shape)
    min_h = min(gt.shape[0], recon.shape[0])
    min_w = min(gt.shape[1], recon.shape[1])
    gt_c = gt[:min_h, :min_w]
    re_c = recon[:min_h, :min_w]

    # RMSE
    rmse = np.sqrt(np.mean((gt_c - re_c)**2))

    # Data range
    data_range = gt_c.max() - gt_c.min()
    if data_range < 1e-10:
        data_range = 1.0

    # PSNR
    mse = np.mean((gt_c - re_c)**2)
    psnr = 10 * np.log10(data_range**2 / (mse + 1e-12))

    # SSIM
    ssim_val = ssim(gt_c, re_c, data_range=data_range)

    # CC (Pearson correlation)
    g = gt_c.flatten() - gt_c.mean()
    r = re_c.flatten() - re_c.mean()
    cc = np.sum(g * r) / (np.sqrt(np.sum(g**2) * np.sum(r**2)) + 1e-12)

    return {
        "PSNR": float(psnr),
        "SSIM": float(ssim_val),
        "RMSE": float(rmse),
        "CC": float(cc),
    }

# ═══════════════════════════════════════════════════════════════════
# 5. VISUALIZATION
# ═══════════════════════════════════════════════════════════════════
def visualize(gt, sinogram, recon, metrics):
    """Plot sinogram, GT, FBP reconstruction, error map."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))

    gt_n = (gt - gt.min()) / (gt.max() - gt.min() + 1e-12)
    re_n = (recon - recon.min()) / (recon.max() - recon.min() + 1e-12)

    im0 = axes[0, 0].imshow(sinogram.T, cmap="gray", aspect="auto",
                             extent=[0, 180, -sinogram.shape[0]//2, sinogram.shape[0]//2])
    axes[0, 0].set_title("Sinogram (neutron transmission)", fontsize=14)
    axes[0, 0].set_xlabel("Angle (degrees)")
    axes[0, 0].set_ylabel("Detector position")
    plt.colorbar(im0, ax=axes[0, 0], fraction=0.046)

    im1 = axes[0, 1].imshow(gt_n, cmap="inferno")
    axes[0, 1].set_title("Ground Truth (μ distribution)", fontsize=14)
    axes[0, 1].axis("off")
    plt.colorbar(im1, ax=axes[0, 1], fraction=0.046)

    im2 = axes[1, 0].imshow(re_n, cmap="inferno")
    axes[1, 0].set_title(
        f"FBP Reconstruction\nPSNR={metrics['PSNR']:.2f} dB, "
        f"SSIM={metrics['SSIM']:.4f}",
        fontsize=12,
    )
    axes[1, 0].axis("off")
    plt.colorbar(im2, ax=axes[1, 0], fraction=0.046)

    error = np.abs(gt_n - re_n)
    im3 = axes[1, 1].imshow(error, cmap="magma")
    axes[1, 1].set_title(f"Absolute Error (RMSE={metrics['RMSE']:.4f} cm⁻¹)", fontsize=12)
    axes[1, 1].axis("off")
    plt.colorbar(im3, ax=axes[1, 1], fraction=0.046)

    plt.tight_layout()
    for p in [os.path.join(RESULTS_DIR, "reconstruction_result.png"),
              os.path.join(ASSETS_DIR, "reconstruction_result.png"),
              os.path.join(ASSETS_DIR, "vis_result.png")]:
        plt.savefig(p, dpi=150, bbox_inches="tight")
    plt.close()

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
print("=" * 60)
print("neutompy — Neutron Tomography Reconstruction")
print("=" * 60)

### 1. Create ground truth

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# 1. Create ground truth
print("[1/4] Creating phantom (attenuation map) ...")
gt = create_phantom(N)

### 2. Forward model

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 2. Forward model
print(f"[2/4] Simulating neutron transmission ({N_ANGLES} angles) ...")
sinogram, sinogram_ideal, angles = forward_neutron_tomo(gt, N_ANGLES, NOISE_LEVEL)

### 3. FBP reconstruction

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 3. FBP reconstruction
print("[3/4] Running Filtered Backprojection ...")
recon = fbp_reconstruct(sinogram, angles)

### 4. Metrics

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# 4. Metrics
metrics = compute_metrics(gt, recon)
print(f"  PSNR = {metrics['PSNR']:.2f} dB")
print(f"  SSIM = {metrics['SSIM']:.4f}")
print(f"  RMSE = {metrics['RMSE']:.4f}")
print(f"  CC   = {metrics['CC']:.4f}")

### 5. Save

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# 5. Save
print("[4/4] Saving results ...")
for d in [RESULTS_DIR, ASSETS_DIR]:
    os.makedirs(d, exist_ok=True)
    np.save(os.path.join(d, "gt_output.npy"), gt)
    np.save(os.path.join(d, "recon_output.npy"), recon)
    np.save(os.path.join(d, "ground_truth.npy"), gt)
    np.save(os.path.join(d, "reconstruction.npy"), recon)
    with open(os.path.join(d, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

visualize(gt, sinogram, recon, metrics)

print("Done ✓")
return metrics

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **neutompy**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=31.54 dB, SSIM=0.6453)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: NeuTomPy toolbox, a Python package for tomographic data processing and reconstruction
- Repository: https://github.com/dmici/NeuTomPy-toolbox
